# Step 2. Register

*With IMPACT-Reg, from KonfAI. This is the only step that uses it.*

**Goal:** bring the moving image onto the fixed one with one command, and get the transformation in OME-Zarr.

With a GPU, replace `--cpu 1` with `--gpu 0`. The first run takes longer: it downloads the preset and the elastix
binary.

In [ ]:
!impact-reg-konfai register Generic_Rigid \
    -f ../output/data/fixed.ome.zarr -m ../output/data/moving.ome.zarr -o ../output/registration --cpu 1 --fields-only \
    --set Predictor.outputs_dataset.DisplacementField.OutputDataset.dataset_filename=Transform:omezarr

[ImpactReg] Registration completed in 16 s
  Preset:    Generic_Rigid (Generic Rigid)
  Fixed:     /path/to/ome-zarr-registration-tutorial/output/data/fixed.ome.zarr
  Moving:    /path/to/ome-zarr-registration-tutorial/output/data/moving.ome.zarr
  Transform: /path/to/ome-zarr-registration-tutorial/output/registration/P000/Transform.ome.zarr
  Record:    /path/to/ome-zarr-registration-tutorial/output/registration/register.json


- **`-f` is the fixed image, `-m` the moving one,** read with their voxel sizes and units.
- **`Generic_Rigid` is a preset:** elastix, rigid, on any machine, in seconds.
- **`--fields-only`** skips the resampled copy of the moving image: this tutorial never writes one.
- **The last line** writes the transformation in OME-Zarr (`Transform:omezarr`). Without it, you get ITK's
  `Transform.h5`.

You get two things:

In [ ]:
!ls ../output/registration ../output/registration/P000

../output/registration:
P000  register.json

../output/registration/P000:
Transform.ome.zarr


- `Transform.ome.zarr`: the transformation, an RFC-5 `affine` of twelve numbers. A deformable preset writes a
  `displacements` field instead, an image of vectors on the fixed grid. Steps 3 and 4 take either.
- `register.json`: what ran, with which versions.

## Which way does it go?

The transformation goes **from the fixed image to the moving one**, even though the moving image is the one that gets
aligned. To draw the aligned image, a reader walks the fixed grid and, for each point $x$, asks where it is in the
moving image:

$$I_{\mathrm{aligned}}(x) = I_{\mathrm{moving}}\big(T_{\mathrm{fixed} \rightarrow \mathrm{moving}}(x)\big)$$

So $T$ takes fixed coordinates and returns moving ones, in micrometers, in z, y, x order. A point of the fixed image
goes through $T$ as it is; a point of the moving image would need the inverse.

## What a preset is

A preset is a folder: a KonfAI app on Hugging Face, in [`VBoussot/ImpactReg`](https://huggingface.co/VBoussot/ImpactReg),
downloaded on first use at the version `impact-reg-konfai` pins. `impact-reg-konfai list` lists them, and
`impact-reg-konfai show Generic_Rigid` says what one does and which parameters it takes. Here is the folder, and the
part of its configuration that is the registration:

In [ ]:
from pathlib import Path

from huggingface_hub import snapshot_download
from impact_reg_konfai import PRESETS_REVISION

preset = Path(snapshot_download("VBoussot/ImpactReg", revision=PRESETS_REVISION, allow_patterns="Generic_Rigid/*")) / "Generic_Rigid"
print(sorted(path.name for path in preset.iterdir()))
print((preset / "Prediction.yml").read_text().split("  Dataset:")[0])

['Evaluation_with_fid.yml', 'Evaluation_with_images.yml', 'Evaluation_with_seg.yml', 'Parameters_Rigid.txt', 'Prediction.yml', 'Uncertainty.yml', 'app.json', 'icon.png', 'requirements.txt']
Predictor:
  Model:
    classpath: impact_reg_konfai.models.elastix:RegistrationNet
    RegistrationNet:
      engine: elastix
      parameter_maps:
      - Parameters_Rigid.txt
      outputs_criterions: None
      max_iterations: 0
      spatial_samples: 2048
      parameter_overrides: []
      moments_init: com
      transform_type: affine



- **`Prediction.yml`** is the KonfAI configuration. Its `Model` block is the registration: the engine (elastix here),
  its parameter map, and the parameters `show` lists. The rest says how the inputs come in and how the result goes
  out: the output `DisplacementField` is written as `Transform:itktransform` by default, and the long `--set` above
  changes that.
- **`Parameters_Rigid.txt`** is elastix's own parameter map. ConvexAdam and FireANTs presets have none: their settings
  are all in the `Model` block.
- **`app.json`** declares the app, its inputs and its output. **`requirements.txt`** is what it installs on first use.
  **`Evaluation_*.yml`** and **`Uncertainty.yml`** are what `eval` and `uncertainty` run.

## Make your own

For one run, `--set NAME=VALUE` changes a parameter of the `Model` block by its name, or any other key by its full
path, as above for the output:

```bash
impact-reg-konfai register Generic_Rigid ... --set max_iterations=50 --set moments_init=cof
```

For a preset of your own, copy the folder, edit it, and point the command to your folder of presets:

```bash
cp -r <the folder above> my_presets/Generic_Rigid_Fast     # then edit my_presets/Generic_Rigid_Fast/Prediction.yml
KONFAI_IMPACTREG_REPO=my_presets impact-reg-konfai register Generic_Rigid_Fast -f ... -m ... -o ...
```

## Try another preset

Only the preset name changes. Steps 3 and 4 run unchanged with any of them.

| Preset | Engine | Transformation | Time on this pair | Correlation | Nuclei mask Dice | Runs on |
|---|---|---|---|---|---|---|
| `Generic_Rigid` | elastix, rigid | `affine` | 15 s, CPU | 0.90 | 0.92 | any machine |
| `Generic_Rigid_BSpline` | elastix, rigid then B-spline | `displacements` | 4 min, CPU | 0.86 | 0.74 | any machine |
| `ConvexAdam_Composite` | ConvexAdam on MIND features | `displacements` | 1 min, GPU | 0.99 | 0.94 | Linux and Windows, no macOS package |
| `FireANTs_SyN` | FireANTs, diffeomorphic | `displacements` | 2 min 30, GPU | 0.99 | 0.94 | a GPU: over 50 min on a CPU |

Deformable is not always better: here the B-spline scores below the rigid it starts from. The scores come from step 4;
before registration, the correlation is -0.06 and the Dice 0. These presets start by matching the two centres of mass
(`moments_init: com`). For two images that cover different parts of a body, `--set moments_init=cof` matches the two
frames instead.

**Next: [Step 3. Write, and read back](3-write.ipynb)**